# 73 - Experimento MPNet + DBSCAN

Este notebook compara embeddings `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` com DBSCAN para agrupar descricoes normalizadas de itens de supermercado. Compara duas sementes de amostragem, 42 e 123, usando a mesma base e periodo do notebook 72. Nao usa NCM nem rotulos humanos; ruido e tratado como resultado explicito.

Execute de cima para baixo em uma sessao limpa. Cada execucao grava relatorios em uma pasta UTC nova dentro de `reports/mpnet_dbscan/`. O primeiro uso do MPNet pode baixar o modelo do Hugging Face.


In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import logging

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from matplotlib.colors import ListedColormap
from sklearn import config_context

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook73")
pd.set_option("display.max_colwidth", 140)


## 1. Parametros

`eps` e o raio maximo de vizinhanca em distancia cosseno; `min_samples` e o minimo de pontos para formar uma regiao densa. Silhouette e calculada somente nos itens nao classificados como ruido. `CONSIDER_FREQUENCY_IN_CLUSTERING` define se a frequencia pondera o DBSCAN e fica False por padrao. A frequencia continua disponivel para ordenar clusters pelo volume e destacar descricoes frequentes em clusters unitarios. A grade e as amostras sao exploratorias; resultados devem ser lidos junto da quantidade de clusters e da fracao de ruido.

O experimento repete a grade com as sementes 42 e 123 para avaliar a sensibilidade a amostragem. MPNet gera embeddings com o tokenizer e a representacao definidos pelo modelo; nao ha um parametro `analyzer` equivalente ao do TF-IDF. Para comparar analyzers, use o notebook 72.

Uma projecao t-SNE reprodutivel de ate `TSNE_MAX_POINTS` descricoes e calculada por amostra; cada combinacao reutiliza essas coordenadas para gerar um PNG.


In [2]:
DATASET_NAME = "2025_NCMs_filtrados-sem-duplicatas"  # Nome da pasta em data/processed; altere apenas este valor para trocar a base.

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Nao encontrei a raiz do repositorio com data/processed/.")
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME
DATASET_FILES = sorted(DATASET_DIR.glob("*.parquet"))
if not DATASET_FILES:
    raise FileNotFoundError(f"Nenhum Parquet encontrado para {DATASET_NAME}: {DATASET_DIR}")
if len(DATASET_FILES) > 1:
    raise ValueError(
        f"Mais de um Parquet encontrado em {DATASET_DIR}; selecione o arquivo/periodo explicitamente: "
        f"{[path.name for path in DATASET_FILES]}"
    )
DATASET_PATH = DATASET_FILES[0]
if not DATASET_PATH.stem.startswith(f"{DATASET_NAME}-"):
    raise ValueError(
        f"O nome do arquivo deve comecar com '{DATASET_NAME}-' para derivar o periodo: "
        f"{DATASET_PATH.name}"
    )
PERIOD_LABEL = DATASET_PATH.stem[len(DATASET_NAME) + 1:]
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
MAX_DESCRIPTIONS = None  # Limite inicial igual ao notebook 72 para controlar o custo do DBSCAN.
CONSIDER_FREQUENCY_IN_CLUSTERING = False  # False: uma observacao por descricao; True: pondera por frequencia.
SAMPLE_SEEDS = [42]
ENCODE_BATCH_SIZE = 64
DBSCAN_N_JOBS = 1
PAIRWISE_WORKING_MEMORY_MIB = 64

EPS_VALUES = [0.03]
MIN_SAMPLES_VALUES = [5, 10, 30]
OUTPUT_ROOT = Path("reports/mpnet_dbscan")


TSNE_MAX_POINTS = 5_000  # Amostra fixa para limitar o custo da projecao.
TSNE_PERPLEXITY = 30
TSNE_MAX_ITER = 1_000
TSNE_RANDOM_STATE = 42

## 2. Fun??es da an?lise

A prepara??o consolida duplicatas sem alterar a frequ?ncia total. O resumo mant?m o r?tulo `-1` para ru?do. A silhouette exclui ru?do e s? ? calculada quando h? pelo menos dois clusters e mais de um item em cada parti??o considerada.


In [3]:
def prepare_items(frame):
    """Limpa descri??es, agrega duplicatas e preserva a frequ?ncia total."""
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY].sum()
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A prepara??o alterou a soma de frequ?ncia.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)


def score_configuration(vectors, labels):
    """Calcula m?tricas de DBSCAN e silhouette sem incluir pontos de ru?do."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    clustered_labels = labels[~noise_mask]
    cluster_count = len(np.unique(clustered_labels))
    score = np.nan
    if 2 <= cluster_count < len(clustered_labels):
        score = float(silhouette_score(vectors[~noise_mask], clustered_labels, metric="cosine"))
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_cosine_without_noise": score,
    }


def summarize_clusters(items, labels):
    """Retorna atribui??es e resumo dos clusters, incluindo a linha de ru?do."""
    result = items.copy()
    result["cluster"] = labels
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary



def compute_tsne_projection(vectors, max_points, perplexity, max_iter, random_state):
    """Calcula uma projecao t-SNE reprodutivel sobre uma amostra de embeddings."""
    point_count = vectors.shape[0]
    if point_count < 3:
        raise ValueError("O t-SNE requer pelo menos tres descricoes.")

    sample_count = min(point_count, max_points)
    if sample_count < 3:
        raise ValueError("TSNE_MAX_POINTS deve ser pelo menos tres.")
    rng = np.random.default_rng(random_state)
    indices = np.sort(rng.choice(point_count, size=sample_count, replace=False))
    effective_perplexity = min(float(perplexity), (sample_count - 1) / 3)
    coordinates = TSNE(
        n_components=2,
        perplexity=effective_perplexity,
        init="pca",
        learning_rate="auto",
        max_iter=max_iter,
        random_state=random_state,
        method="barnes_hut",
    ).fit_transform(vectors[indices])
    return indices, coordinates


def save_tsne_plot(coordinates, labels, run_id, output_path):
    """Salva um PNG t-SNE colorido pelos clusters, com o ruido em cinza."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_ids = np.unique(labels[~noise_mask])
    if cluster_ids.size:
        palette = plt.get_cmap("tab20").colors
        cluster_cmap = ListedColormap([
            palette[index % len(palette)] for index in range(len(cluster_ids))
        ])
        cluster_codes = np.searchsorted(cluster_ids, labels[~noise_mask])
    else:
        cluster_cmap = None
        cluster_codes = np.empty(0, dtype=int)

    figure, axis = plt.subplots(figsize=(11, 8), constrained_layout=True)
    if noise_mask.any():
        axis.scatter(
            coordinates[noise_mask, 0], coordinates[noise_mask, 1],
            c="#b8b8b8", s=8, alpha=0.3, linewidths=0,
        )
    if cluster_ids.size:
        axis.scatter(
            coordinates[~noise_mask, 0], coordinates[~noise_mask, 1],
            c=cluster_codes, cmap=cluster_cmap, vmin=-0.5,
            vmax=len(cluster_ids) - 0.5, s=9, alpha=0.75, linewidths=0,
        )
    axis.set_title(
        f"t-SNE: {run_id}\n{len(cluster_ids)} clusters; "
        f"{int(noise_mask.sum()):,} descricoes como ruido"
    )
    axis.set_xlabel("t-SNE 1")
    axis.set_ylabel("t-SNE 2")
    axis.text(
        0.01, 0.01,
        "Ruido em cinza; as cores se repetem a cada 20 clusters.",
        transform=axis.transAxes, fontsize=8, color="#555555",
    )
    figure.savefig(output_path, dpi=160)
    plt.close(figure)


## 3. Carregar, preparar e gerar embeddings

A amostra e selecionada separadamente para cada semente. O modelo MPNet transforma cada amostra em embeddings normalizados, que serao reutilizados em toda a grade de DBSCAN daquela semente.


In [4]:
raw_items = pd.read_parquet(DATASET_PATH)
required_columns = {DESCRIPTION_NORMALIZED, FREQUENCY}
missing_columns = required_columns - set(raw_items.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")
base_items = prepare_items(raw_items)
logger.info("Descricoes disponiveis apos preparacao: %d", len(base_items))
logger.info("Carregando modelo %s", MODEL_NAME)
encoder = SentenceTransformer(MODEL_NAME)

items_by_seed = {}
vectors_by_seed = {}
tsne_by_seed = {}
for sample_seed in SAMPLE_SEEDS:
    sampled_items = base_items
    if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
        sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=sample_seed)
    sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    vectors = encoder.encode(
        sampled_items[DESCRIPTION_NORMALIZED].astype(str).tolist(),
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    logger.info("Calculando t-SNE para seed=%d", sample_seed)
    tsne_indices, tsne_coordinates = compute_tsne_projection(
        vectors, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE + sample_seed,
    )
    tsne_by_seed[sample_seed] = {"indices": tsne_indices, "coordinates": tsne_coordinates}
    items_by_seed[sample_seed] = sampled_items
    vectors_by_seed[sample_seed] = vectors
    logger.info(
        "Seed %d: %d descricoes, embeddings %s, frequencia total %d",
        sample_seed, len(sampled_items), vectors.shape, sampled_items[FREQUENCY].sum(),
    )

display(pd.DataFrame([
    {
        "sample_seed": sample_seed,
        "descricoes": len(items_by_seed[sample_seed]),
        "dimensoes_embedding": vectors_by_seed[sample_seed].shape[1],
        "frequencia_total": int(items_by_seed[sample_seed][FREQUENCY].sum()),
    }
    for sample_seed in SAMPLE_SEEDS
]))


2026-10-15 10:28:01,830 INFO Descricoes disponiveis apos preparacao: 149149
2026-10-15 10:28:01,830 INFO Carregando modelo sentence-transformers/paraphrase-multilingual-mpnet-base-v2
2026-10-15 10:28:01,830 INFO No device provided, using cpu
2026-10-15 10:28:02,252 INFO HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-10-15 10:28:02,400 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-15 10:28:02,412 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-15 10:28:02,435 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/modules.json?%2Fsentence-transformers%2Fparaphrase-multilingual-mpnet-base-v2%2Fresolve%2Fma

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-10-15 10:28:07,395 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
2026-10-15 10:28:07,603 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-15 10:28:07,751 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-15 10:28:12,025 INFO HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2 "HTTP/1.1 200 OK"
2026-10-15 10:28:12,199 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-15 10:28:12,224 INFO HTTP Request: HEAD https:

Batches:   0%|          | 0/2331 [00:00<?, ?it/s]

2026-10-15 11:29:45,380 INFO Calculando t-SNE para seed=42
2026-10-15 11:30:10,381 INFO Seed 42: 149149 descricoes, embeddings (149149, 768), frequencia total 868288


,sample_seed,descricoes,dimensoes_embedding,frequencia_total
0,42,149149,768,868288


## 4. Ajustar DBSCAN e comparar parametros

DBSCAN usa distancia cosseno nos vetores densos do MPNet. `CONSIDER_FREQUENCY_IN_CLUSTERING` controla o `sample_weight`; por padrao, cada descricao unica conta uma vez. Silhouette e fracao de ruido tambem contam descricoes unicas. Os embeddings de cada semente sao reutilizados nos ajustes correspondentes.

A projecao t-SNE e calculada uma vez por semente sobre uma amostra fixa; cada combinacao de `eps` e `min_samples` reutiliza as mesmas coordenadas no PNG.


In [ ]:
records = []
labels_by_run = {}
items_by_run = {}
for sample_seed in SAMPLE_SEEDS:
    items = items_by_seed[sample_seed]
    vectors = vectors_by_seed[sample_seed]
    for eps in EPS_VALUES:
        for min_samples in MIN_SAMPLES_VALUES:
            experiment_id = f"seed{sample_seed}_eps{eps:g}_min_samples{min_samples}"
            logger.info("Ajustando DBSCAN %s", experiment_id)
            model = DBSCAN(
                eps=eps, min_samples=min_samples, metric="cosine",
                algorithm="brute", n_jobs=DBSCAN_N_JOBS,
            )
            with config_context(working_memory=PAIRWISE_WORKING_MEMORY_MIB):
                sample_weight = (
                    items[FREQUENCY].to_numpy(dtype=float)
                    if CONSIDER_FREQUENCY_IN_CLUSTERING else None
                )
                model.fit(vectors, sample_weight=sample_weight)
            labels = model.labels_
            metrics = score_configuration(vectors, labels)
            records.append({
                "run_id": experiment_id, "sample_seed": sample_seed,
                "eps": eps, "min_samples": min_samples,
                "frequency_weighted": CONSIDER_FREQUENCY_IN_CLUSTERING, **metrics,
            })
            labels_by_run[experiment_id] = labels
            items_by_run[experiment_id] = items
results = pd.DataFrame(records).sort_values(
    ["silhouette_cosine_without_noise", "noise_fraction"],
    ascending=[False, True], na_position="last",
).reset_index(drop=True)
display(results)


2026-10-15 11:30:10,441 INFO Ajustando DBSCAN seed42_eps0.03_min_samples5


## 5. Revisar um resultado e salvar relat?rios

A sele??o autom?tica usa a maior silhouette dispon?vel; confira tamb?m quantos clusters foram formados e a fra??o de ru?do antes de interpretar. Todos os resultados da grade s?o salvos, e os arquivos da execu??o n?o sobrescrevem resultados anteriores.

Os graficos ficam em `tsne_plots/`, um PNG por combinacao, e as coordenadas da amostra em `tsne_coordinates/`, dentro da pasta da execucao.


In [ ]:
SELECTED_RUN_ID = results.iloc[0]["run_id"]
selected = results.set_index("run_id").loc[SELECTED_RUN_ID]
selected_items = items_by_run[SELECTED_RUN_ID]
selected_labels = labels_by_run[SELECTED_RUN_ID]
assignments, cluster_summary = summarize_clusters(selected_items, selected_labels)
cluster_summary = cluster_summary.sort_values("frequencia_total", ascending=False).reset_index(drop=True)
cluster_sizes = assignments.loc[assignments["cluster"].ne(-1)].groupby("cluster")[DESCRIPTION_NORMALIZED].size()
singleton_clusters = cluster_sizes[cluster_sizes.eq(1)].index
frequent_singletons = assignments[assignments["cluster"].isin(singleton_clusters)].sort_values(
    FREQUENCY, ascending=False
)
frequent_noise = assignments[assignments["cluster"].eq(-1)].sort_values(
    FREQUENCY, ascending=False
)
display(cluster_summary.head(20))
display(frequent_singletons.head(20))
display(frequent_noise.head(20))

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)
results.to_csv(output_dir / "metrics.csv", index=False)
assignments.to_csv(output_dir / "assignments.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(output_dir / "cluster_summary.csv", index=False, encoding="utf-8-sig")
frequent_singletons.to_csv(
    output_dir / "frequent_singleton_descriptions.csv", index=False, encoding="utf-8-sig"
)
frequent_noise.to_csv(
    output_dir / "frequent_noise_descriptions.csv", index=False, encoding="utf-8-sig"
)

# Uma projecao por amostra e reutilizada em todas as combinacoes de eps/min_samples.
tsne_coordinates_dir = output_dir / "tsne_coordinates"
tsne_plots_dir = output_dir / "tsne_plots"
tsne_coordinates_dir.mkdir()
tsne_plots_dir.mkdir()
tsne_artifacts = {}
for sample_seed, projection in tsne_by_seed.items():
    indices = projection["indices"]
    coordinates = projection["coordinates"]
    coordinate_frame = items_by_seed[sample_seed].iloc[indices][
        [DESCRIPTION_NORMALIZED, FREQUENCY]
    ].reset_index(drop=True)
    coordinate_frame["tsne_1"] = coordinates[:, 0]
    coordinate_frame["tsne_2"] = coordinates[:, 1]
    coordinate_name = f"seed{sample_seed}_coordinates.csv"
    coordinate_frame.to_csv(
        tsne_coordinates_dir / coordinate_name, index=False, encoding="utf-8-sig",
    )
    tsne_artifacts[f"seed{sample_seed}"] = {
        "coordinates_file": f"tsne_coordinates/{coordinate_name}",
        "sample_size": len(indices),
    }

for run_id, labels in labels_by_run.items():
    sample_seed = int(results.loc[results["run_id"].eq(run_id), "sample_seed"].iloc[0])
    projection = tsne_by_seed[sample_seed]
    plot_labels = np.asarray(labels)[projection["indices"]]
    plot_path = tsne_plots_dir / f"{run_id}.png"
    save_tsne_plot(projection["coordinates"], plot_labels, run_id, plot_path)
    tsne_artifacts[f"plot:{run_id}"] = str(plot_path.relative_to(output_dir))

logger.info("Imagens t-SNE salvas: %d", len(labels_by_run))
run_metadata = {
    "dataset_name": DATASET_NAME, "period": PERIOD_LABEL,
    "dataset_path": str(DATASET_PATH), "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seeds": SAMPLE_SEEDS, "model_name": MODEL_NAME,
    "embedding_normalized": True,
    "frequency_weighted": CONSIDER_FREQUENCY_IN_CLUSTERING,
    "selected_run_id": SELECTED_RUN_ID, "eps_values": EPS_VALUES,
    "min_samples_values": MIN_SAMPLES_VALUES,
    "tsne": {
        "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
        "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
        "artifacts": tsne_artifacts,
    },
}
(output_dir / "run.json").write_text(json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8")
logger.info("Relatorios e imagens t-SNE salvos em %s", output_dir)


,cluster,descricoes_unicas,frequencia_total
0,-1,253912,802300
719,718,977,999
1315,1314,529,529
306,305,45,103
570,569,33,421
...,...,...,...
16,15,1,432
15,14,1,100
13,12,1,579
12,11,1,337


2026-10-14 22:50:50,415 INFO Relatorios salvos em reports\mpnet_dbscan\202501-202512_20261015T015049Z


### Interpreta??o

- `noise_fraction` alta indica que muitas descri??es ficaram fora de regi?es densas para os par?metros escolhidos.
- Silhouette deve ser interpretada junto da quantidade de clusters e da fra??o de ru?do; uma pontua??o alta pode refletir poucos pontos classificados.
- A compara??o com TF-IDF ? explorat?ria: os dois m?todos usam a mesma amostra, mas representam similaridade de formas diferentes.
- DBSCAN n?o fornece uma opera??o direta para atribuir novas descri??es a clusters ap?s o ajuste e n?o define categorias sem revis?o humana.
